# Online Dynamic Batching single-GPU demo

This notebook runs a tiny synthetic variable-length training loop and compares fixed batching with ODB. It does not require private data.

## Install

Install ODB from PyPI before running the notebook locally:

```bash
pip install online-dynamic-batching
```

In [ ]:
import math
import time

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

import odb

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
class SyntheticSequenceDataset(Dataset):
    def __init__(self, num_samples=1024, vocab_size=256, seed=7):
        generator = torch.Generator().manual_seed(seed)
        short = torch.randint(64, 384, (int(num_samples * 0.72),), generator=generator)
        medium = torch.randint(384, 1400, (int(num_samples * 0.22),), generator=generator)
        long = torch.randint(1400, 4096, (num_samples - short.numel() - medium.numel(),), generator=generator)
        self.lengths = torch.cat([short, medium, long]).tolist()
        self.vocab_size = vocab_size

    def __len__(self):
        return len(self.lengths)

    def __getitem__(self, idx):
        length = int(self.lengths[idx])
        values = (torch.arange(length, dtype=torch.long) + idx) % self.vocab_size
        return {'input_ids': values, 'labels': values.clone()}


def collate(batch):
    max_len = max(item['input_ids'].numel() for item in batch)
    input_ids = torch.zeros(len(batch), max_len, dtype=torch.long)
    labels = torch.zeros(len(batch), max_len, dtype=torch.long)
    attention_mask = torch.zeros(len(batch), max_len, dtype=torch.bool)
    for row, item in enumerate(batch):
        length = item['input_ids'].numel()
        input_ids[row, :length] = item['input_ids']
        labels[row, :length] = item['labels']
        attention_mask[row, :length] = True
    return {'input_ids': input_ids, 'labels': labels, 'attention_mask': attention_mask}


class TinyLM(nn.Module):
    def __init__(self, vocab_size=256, hidden_size=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.proj = nn.Linear(hidden_size, vocab_size)

    def forward(self, input_ids, labels, attention_mask):
        logits = self.proj(self.embedding(input_ids))
        loss = nn.functional.cross_entropy(logits.flatten(0, 1), labels.flatten(0, 1), reduction='none')
        loss = loss.view_as(labels)
        return (loss * attention_mask).sum() / attention_mask.sum().clamp_min(1)

In [ ]:
def run_epoch(name, dataloader, model):
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    start = time.perf_counter()
    samples = real_tokens = padded_tokens = steps = 0
    final_loss = math.nan

    for batch in dataloader:
        info = odb.pop_step_info(batch, loss_scaling='exact')
        batch = {key: value.to(device) for key, value in batch.items()}
        loss = model(**batch) * info.loss_scale
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        local_samples = int(batch['input_ids'].shape[0])
        samples += int(info.all_samples_this_step or local_samples)
        real_tokens += int(batch['attention_mask'].sum().item())
        padded_tokens += int(batch['input_ids'].numel())
        steps += 1
        final_loss = float(loss.detach().cpu().item())

    seconds = time.perf_counter() - start
    return {
        'name': name,
        'samples_per_second': samples / seconds,
        'real_tokens_per_second': real_tokens / seconds,
        'padding_ratio': 1.0 - real_tokens / max(padded_tokens, 1),
        'steps': steps,
        'loss': final_loss,
    }

In [ ]:
dataset = SyntheticSequenceDataset(num_samples=1024)

fixed_loader = DataLoader(dataset, batch_size=8, shuffle=False, num_workers=2, prefetch_factor=16, collate_fn=collate)
odb_loader = odb.ODBDataLoader(
    dataset,
    token_budget=8192,
    batch_size=1,
    shuffle=False,
    num_workers=2,
    prefetch_factor=16,
    collate_fn=collate,
    loss_scaling='exact',
)

fixed_model = TinyLM().to(device)
odb_model = TinyLM().to(device)
odb_model.load_state_dict(fixed_model.state_dict())

fixed_result = run_epoch('fixed', fixed_loader, fixed_model)
odb_result = run_epoch('odb', odb_loader, odb_model)
fixed_result, odb_result